## Import Libraries

In [5]:
!pip install -q keras-tuner

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import keras_tuner as kt

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


## Load Dataset

In [6]:
df = pd.read_csv('bengaluru_house_prices.csv')

df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


## Data Preprocessing & Cleaning

In [7]:
df = df.drop(columns=['area_type', 'availability', 'society', 'balcony'])

df = df.dropna()

df['bhk'] = df['size'].apply(lambda x: int(x.split(' ')[0]))
df = df.drop(columns=['size'])

def convert_sqft_to_num(x):
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)
df = df.dropna()

location_stats = df['location'].value_counts()
locations_less_than_10 = location_stats[location_stats <= 10]
df['location'] = df['location'].apply(lambda x: 'other' if x in locations_less_than_10 else x)

df = pd.get_dummies(df, columns=['location'], drop_first=True, dtype=int)

df.head()

,total_sqft,bath,price,bhk,location_1st Block Jayanagar,location_1st Phase JP Nagar,location_2nd Phase Judicial Layout,location_2nd Stage Nagarbhavi,location_5th Block Hbr Layout,location_5th Phase JP Nagar,...,location_Vishveshwarya Layout,location_Vishwapriya Layout,location_Vittasandra,location_Whitefield,location_Yelachenahalli,location_Yelahanka,location_Yelahanka New Town,location_Yelenahalli,location_Yeshwanthpur,location_other
0,1056.0,2.0,39.07,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2600.0,5.0,120.00,4,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1440.0,2.0,62.00,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1521.0,3.0,95.00,3,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1200.0,2.0,51.00,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Split Data & Feature Scaling

In [8]:
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train_scaled.shape}")

X_train shape: (10560, 243)


## Build ANN Model

In [9]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        31,232 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,601 (162.50 KB)

 Trainable params: 41,601 (162.50 KB)

 Non-trainable params: 0 (0.00 B)

## Train the Model

In [10]:
history = model.fit(X_train_scaled, y_train,
                    validation_split=0.2,
                    epochs=60,
                    batch_size=32,
                    verbose=1)

Epoch 1/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - loss: 19434.4160 - mae: 66.2796 - val_loss: 17593.9414 - val_mae: 49.3260
Epoch 2/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 13740.2627 - mae: 47.5411 - val_loss: 16753.4297 - val_mae: 44.4317
Epoch 3/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 12872.0469 - mae: 45.7690 - val_loss: 16983.9062 - val_mae: 42.9884
Epoch 4/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 12665.2588 - mae: 44.1493 - val_loss: 16175.8789 - val_mae: 41.3360
Epoch 5/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 12448.7900 - mae: 43.0264 - val_loss: 15785.5342 - val_mae: 43.9003
Epoch 6/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 12089.8701 - mae: 42.9557 - val_loss: 15474.9043 - val_mae: 40.7461
Epoch 7/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 11597.3535 - mae: 42.0205 - val_loss: 15412.0850 - val_mae: 42.2232
Epoch 8/60
264/264 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 11861.0332 - mae: 42.8547 - val_loss: 150

## Evaluate the Model

## Hyperparameter Tuning (Find Best Parameters)

In [11]:
def build_regression_model(hp):
    model = Sequential()

    hp_units = hp.Int('units', min_value=64, max_value=256, step=64)
    model.add(Input(shape=(X_train_scaled.shape[1],)))
    model.add(Dense(units=hp_units, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='mse',
        metrics=['mae']
    )

    return model

tuner = kt.RandomSearch(
    build_regression_model,
    objective='val_loss',
    max_trials=5,
    executions_per_trial=1,
    directory='my_dir',
    project_name='house_prices_tuning'
)

tuner.search(X_train_scaled, y_train, epochs=10, validation_split=0.2)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best Units: {best_hps.get('units')}")
print(f"Best Learning Rate: {best_hps.get('learning_rate')}")

Trial 5 Complete [00h 00m 17s]
val_loss: 20556.869140625

Best val_loss So Far: 12835.5478515625
Total elapsed time: 00h 01m 16s
Best Units: 192
Best Learning Rate: 0.01


In [12]:
mse, mae = model.evaluate(X_test_scaled, y_test)
print(f"Test Mean Absolute Error (MAE): {mae:.2f}")

83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 7387.7271 - mae: 37.0895
Test Mean Absolute Error (MAE): 37.09
